In [16]:
import warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight


from sklearn.preprocessing import LabelEncoder

In [17]:

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")

# Load prepared train/test data
train_df = pd.read_csv("C:/Users/BLACKBOX/Desktop/Research Data/Iyenshi/Coding Assignment/data sets/scaled_train_set.csv")
test_df = pd.read_csv("C:/Users/BLACKBOX/Desktop/Research Data/Iyenshi/Coding Assignment/data sets/scaled_test_set.csv")

# BUG FIX: validate required column exists (mirrors TreeBasedModels.ipynb pattern)
if "Attack Type_encoded" not in train_df.columns or "Attack Type_encoded" not in test_df.columns:
    raise ValueError("The train/test files must already contain 'Attack Type_encoded'.")

In [18]:
feature_cols = [c for c in train_df.columns if c not in ["Attack Type", "Attack Type_encoded"]]

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
y_train = train_df["Attack Type_encoded"]
y_test = test_df["Attack Type_encoded"]

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# The dataset is already encoded and scaled, so we only need to convert it to tensors.
X_train_tensor = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.to_numpy(), dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)

print("Training tensors:", X_train_tensor.shape, y_train_tensor.shape)
print("Test tensors:", X_test_tensor.shape, y_test_tensor.shape)

# Create PyTorch data loaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

Training tensors: torch.Size([3713654, 29]) torch.Size([3713654])
Test tensors: torch.Size([928414, 29]) torch.Size([928414])


In [20]:
weights = compute_class_weight(
    class_weight="balanced", classes=np.unique(y_train), y=y_train
)

In [21]:
# BUG FIX: device is determined here so the criterion weight tensor can be moved
# to the correct device during training, preventing CPU/GPU tensor mismatch.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights_tensor = torch.tensor(
    weights,
    dtype=torch.float32
).to(device)  # BUG FIX: move weights to same device as model outputs

criterion = nn.CrossEntropyLoss(weight=weights_tensor)

In [22]:
class MLP(nn.Module):

    def __init__(self, input_dim, num_classes):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim,256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128,64),
            nn.ReLU(),

            nn.Linear(64,num_classes)
        )

    def forward(self,x):
        return self.network(x)

In [23]:
class CNN1D(nn.Module):

    def __init__(self,input_dim,num_classes):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv1d(1,32,3,padding=1),
            nn.ReLU(),

            nn.Conv1d(32,64,3,padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.fc = nn.Sequential(

            nn.Linear(64,64),
            nn.ReLU(),

            nn.Linear(64,num_classes)
        )

    def forward(self,x):

        x = x.unsqueeze(1)

        x = self.conv(x)

        x = x.squeeze(-1)

        return self.fc(x)

In [24]:
def train_model(model, epochs=20):

    # BUG FIX: use the shared `device` variable defined in the criterion cell
    # so the model and criterion weights are always on the same device.
    model.to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=1e-3
    )

    for epoch in range(epochs):

        model.train()

        running_loss = 0

        for x,y in train_loader:

            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            outputs = model(x)

            loss = criterion(outputs,y)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        print(
            f"Epoch {epoch+1} Loss {running_loss:.3f}"
        )

    return model

In [25]:
# BUG FIX: evaluate_nn now returns a structured metrics dict (matching
# TreeBasedModels pattern) so results can be appended to results.csv.
def evaluate_nn(model, model_name):

    model.eval()

    preds = []

    actual = []

    with torch.no_grad():

        for x,y in test_loader:

            x = x.to(device)

            outputs = model(x)

            prediction = outputs.argmax(1).cpu().numpy()

            preds.extend(prediction)

            actual.extend(y.numpy())

    print(classification_report(actual, preds))

    print(confusion_matrix(actual, preds))

    # Build metrics dict matching the CSV schema used by TreeBasedModels
    metrics = {
        "model": model_name,
        "status": "success",
        "accuracy": accuracy_score(actual, preds),
        "precision_macro": precision_score(actual, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(actual, preds, average="macro", zero_division=0),
        "f1_macro": f1_score(actual, preds, average="macro", zero_division=0),
        "error": "",
    }

    return preds, metrics

In [26]:
from pathlib import Path
import numpy as np
import pandas as pd

save_dir = Path("saved/models")
save_dir.mkdir(parents=True, exist_ok=True)

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)
results_file = results_dir / "results.csv"

classes = np.unique(y_train)
num_classes = len(classes)


# BUG FIX: check file existence BEFORE writing (not after), so the header
# flag is evaluated correctly on the very first call.
def append_result_row(metrics):
    write_header = not results_file.exists()
    row = pd.DataFrame([metrics])
    row.to_csv(results_file, mode="a", header=write_header, index=False)

In [27]:
mlp = MLP(
    X_train.shape[1],
    len(classes)
)

mlp = train_model(mlp)

# BUG FIX: pass model name, capture metrics, and save to CSV
_, mlp_metrics = evaluate_nn(mlp, "MLP")
append_result_row(mlp_metrics)
print(f"\nMLP results saved to {results_file}")

torch.save(
    mlp.state_dict(),
    save_dir/"mlp.pth"
)

Epoch 1 Loss 28076.735
Epoch 2 Loss 23209.646
Epoch 3 Loss 22293.898
Epoch 4 Loss 21745.523
Epoch 5 Loss 21396.972
Epoch 6 Loss 21074.401
Epoch 7 Loss 20804.250
Epoch 8 Loss 20587.113
Epoch 9 Loss 20424.898
Epoch 10 Loss 20284.017
Epoch 11 Loss 20187.605
Epoch 12 Loss 20083.867
Epoch 13 Loss 19984.343
Epoch 14 Loss 19920.627
Epoch 15 Loss 19891.484
Epoch 16 Loss 19809.584
Epoch 17 Loss 19793.319
Epoch 18 Loss 19737.213
Epoch 19 Loss 19686.216
Epoch 20 Loss 19634.140
              precision    recall  f1-score   support

           0       0.22      0.86      0.35     28000
           1       0.97      0.93      0.95    156761
           2       0.98      0.83      0.90    662033
           3       0.61      0.80      0.69     81620

    accuracy                           0.84    928414
   macro avg       0.69      0.86      0.72    928414
weighted avg       0.92      0.84      0.87    928414

[[ 24192      0   1793   2015]
 [  3547 146268   5308   1638]
 [ 69480   5011 548892  38650]
 

In [28]:
cnn = CNN1D(
    X_train.shape[1],
    len(classes)
)

cnn = train_model(cnn)

# BUG FIX: pass model name, capture metrics, and save to CSV
_, cnn_metrics = evaluate_nn(cnn, "CNN1D")
append_result_row(cnn_metrics)
print(f"\nCNN1D results saved to {results_file}")

torch.save(
    cnn.state_dict(),
    save_dir/"cnn1d.pth"
)

Epoch 1 Loss 26839.139
Epoch 2 Loss 19154.904
Epoch 3 Loss 18250.799
Epoch 4 Loss 17677.396
Epoch 5 Loss 17275.239
Epoch 6 Loss 16981.157
Epoch 7 Loss 16780.711
Epoch 8 Loss 16662.893
Epoch 9 Loss 16450.681
Epoch 10 Loss 16362.202
Epoch 11 Loss 16480.352
Epoch 12 Loss 16406.467
Epoch 13 Loss 16274.142
Epoch 14 Loss 16428.162
Epoch 15 Loss 16346.461
Epoch 16 Loss 16300.781
Epoch 17 Loss 16160.225
Epoch 18 Loss 16275.164
Epoch 19 Loss 16156.391
Epoch 20 Loss 16056.323
              precision    recall  f1-score   support

           0       0.58      0.81      0.68     28000
           1       0.96      0.95      0.96    156761
           2       0.99      0.83      0.90    662033
           3       0.42      0.92      0.58     81620

    accuracy                           0.86    928414
   macro avg       0.74      0.88      0.78    928414
weighted avg       0.92      0.86      0.88    928414

[[ 22800      8    993   4199]
 [  1771 148461   3354   3175]
 [ 10837   5425 549638  96133]
 

In [29]:
# Final comparison — reload CSV and print sorted summary
if results_file.exists():
    results_df = pd.read_csv(results_file)
    results_df = results_df.sort_values(by="f1_macro", ascending=False, na_position="last").reset_index(drop=True)
    print("\nModel comparison (sorted by F1 Macro)")
    print(results_df)
    results_df.to_csv(results_file, index=False)
    print(f"\nAll results saved to: {results_file}")


Model comparison (sorted by F1 Macro)
           model   status  accuracy  precision_macro  recall_macro  f1_macro  \
0        XGBoost  success  0.951093         0.964300      0.838673  0.890479   
1  Random Forest  success  0.895136         0.776025      0.919842  0.794901   
2          CNN1D  success  0.857462         0.737760      0.878169  0.777912   
3       LightGBM  success  0.866145         0.722605      0.906769  0.756747   
4            MLP  success  0.844817         0.693060      0.855598  0.721259   

   error  
0    NaN  
1    NaN  
2    NaN  
3    NaN  
4    NaN  

All results saved to: results\results.csv
